In [3]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scaling_llms.models import GPTConfig, GPTModel


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
DATASET_KWARGS = dict(
    dataset_name="Salesforce/wikitext",
    dataset_config="wikitext-103-raw-v1",
    train_split="train",
    eval_split="validation",
    tokenizer_name="gpt2_tiktoken",
    text_field="text"
)

DATALOADER_KWARGS = dict(
    seq_len=512,
    train_batch_size=8,
    eval_batch_size=8,
    start_sample_idx=0,
    seed=42           # outer seed; inner seed swept below
)

GPT_HPARAMS = dict(
    # n_embd TO DEFINE IN WIDTH-SWEEP BELOW
    n_layer=6,           # depth fixed throughout
    n_head=4,
    mlp_type="standard_gelu",
    norm_type="rmsnorm",
    pos_encoding_type="rotary",
    tied_embeddings=False,
    attn_bias=False,
    lm_head_bias=False,
    mlp_bias=False,
    attn_pdrop=0.0,
    resid_pdrop=0.0,
)

TRAINER_KWARGS = dict(
    num_steps=0,          # forward-pass only at init — no training needed
    lr=1e-3,              # fixed, irrelevant since num_steps=0
    lr_schedule="none",
    warmup_steps=0,
    weight_decay=0.0,
    grad_clip_norm=None,
    precision="bf16",
    device="cuda",
    eval_log_freq=1,
    net_log_freq=1,
)


In [ ]:


TRAINER_KWARGS = dict(
    num_steps=50,
    beta1=0.9,
    beta2=0.95,
    weight_decay=0.0,
    accum_steps=1, 
    grad_clip_norm=None,
    lr_schedule="none",
    warmup_steps=0,
    min_lr_ratio=0.0,
    enable_tb=False,
    eval_log_freq=5,
    net_log_freq=5,
    sys_log_freq=10,
    ckpt_log_freq=0,
    enable_cuda_timer=False,
)


BASE_GPT_HPARAMS = dict(
    n_layer=12,
    n_head=12,
    d_model=768,
    d_head=64,
    d_mlp=3072,
    attn_pdrop=0.1,
    resid_pdrop=0.1,
    embd_pdrop=0.1,
)
# CONSTANT_OPTIMIZATION_KWARGS = {
#     "lr": 3e-4,
#     "beta1": 0.9,
#     "beta2": 0.95,
#     "precision": "bf16",
#     "device": "cuda",
# }

We fix a given width and initialize multiple models using $N$ different random seeds.  
Each model is trained for a small number of steps. At every training step, we record the activations of every layer:

$$
(\text{width},\ \text{step},\ \text{layer}) \;\rightarrow\; [a_1, a_2, \ldots, a_N].
$$

Then, for each such collection we compute the absolute mean across seeds:

$$(\text{width},\ \text{step},\ \text{layer}) \;\rightarrow\; \operatorname{mean\_abs\_act}
= \frac{1}{N}\sum_{i=1}^N |a_i|.$$

Finally, for every step and every layer, we plot  
$$
\operatorname{mean\_abs\_act} \text{vs.} \text{width},
$$  
and generate these plots for both **SP** and **μP** to compare their behaviors.


**Note:** It suffices to use a small subset of the training dataset for this experiment.


---

SP:
- Final layer's weights receive updates that scale as O(1/sqrt(n))
- Thus, after init logits are expected to be unstable with increasing width
- Mean Abs of Logits for higher widths is expected to diverge as training progresses

muP:
- Final layer's weights receive O(1) updates
- Mean Abs of Logits is exected to be roughly constant for across widths and training steps.

In [ ]:
# FIXED = {
#     "n_layer": 4,           # depth fixed throughout
#     "n_head": 4,            # fixed — so d_head scales with width
#     "seq_len": 256,         # short enough for fast runs
#     "mlp_type": "standard_gelu",
#     "total_tokens": 100_000_000,  # 100M tokens per run — enough to see loss differences
#     "batch_tokens": 131072,
#     "mup_base_width": 128,  # your proxy
# }

SEEDS = [1, 12, 123, 1234, 12345]  # 5 random seeds for each width
WIDTHS = [128, 256, 512, 1024, 2048]  # proxy + 3 transfer targets
# LR_SWEEP = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]  # 6 values, 1.5 orders of magnitude
LOG2LRS = np.linspace(-13, -1, 8, endpoint=False)
LR_SWEEP = [2**log2lr for log2lr in LOG2LRS]

# for coord check
num_steps = 5 
iter_mode = "single-batch" 


In [ ]:
# Pseudocode for what you're launching
for width in WIDTHS:
    for lr in LR_SWEEP:
        cfg = GPTConfig(
            n_embd=width,
            mup_base_width=128,  # always the proxy width
            n_head=4,
            n_layer=4,
            ...
        )
        model = GPTModel(cfg)
        groups = model.get_param_groups(base_lr=lr, weight_decay=0.1)
        optimizer = torch.optim.AdamW(groups)
        train(model, optimizer, tokens=100_000_000)
        log(width=width, lr=lr, val_bpb=evaluate(model))